[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t03_shadows.ipynb)

# Tangent was born as a shadow: all six trig functions

Stand a stick in the sun: its shadow's length is a cotangent of the Sun's height. A stick fixed flat to a wall casts
a tangent. Al-Biruni's book on shadows (about 1021) gathered these tables. Thomas Fincke named the "tangent" and the
"secant" in 1583: the touching line and the cutting line.

Here the six functions arrive one at a time, as lengths that the **turning arrow** makes on two lines, x = 1 and y = 1.

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Two sticks, two shadows

A stick one unit tall, standing on the ground, casts a shadow on the ground. A stick one unit long, sticking straight
out of a wall, casts a shadow down the wall. Raise the Sun and watch both.

The ground shadow is the **cotangent** of the Sun's height. The wall shadow is the **tangent**.

In [ ]:
def two_sticks(sun_deg=35.0):
    a = math.radians(sun_deg)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [2.3, 1]})
    g = 1 / math.tan(a)
    ax.plot([0, 0], [0, 1], color=CHALK, lw=5)
    ax.plot([0, g], [0, 0], color=CORAL, lw=8, solid_capstyle="butt")
    ax.plot([0, g], [1, 0], color=DIM, ls="--", lw=1.2)
    ax.set_xlim(-0.3, 4.2); ax.set_ylim(-0.3, 1.6); ax.set_aspect("equal"); ax.grid(False)
    ax.set_title(f"stick on the ground:\nshadow = cot {sun_deg:.0f}° = {g:.3f}", fontsize=13)
    w = math.tan(a)
    bx.plot([0, 0], [-2.9, 1.3], color=DIM, lw=3)                       # the wall
    bx.plot([0, 1], [1, 1], color=CHALK, lw=5)                           # the stick
    bx.plot([0, 0], [1, 1 - w], color=TEAL, lw=8, solid_capstyle="butt")
    bx.plot([1, 0], [1, 1 - w], color=DIM, ls="--", lw=1.2)
    bx.set_xlim(-0.5, 1.7); bx.set_ylim(-3.0, 1.5); bx.set_aspect("equal"); bx.grid(False)
    bx.set_title(f"stick on a wall:\nshadow = tan {sun_deg:.0f}° = {w:.3f}", fontsize=13)
    plt.tight_layout(); plt.show()

interact(two_sticks, sun_deg=FloatSlider(value=35, min=10, max=75, step=1, description="Sun (°)"));

**Notice.** As the Sun climbs, the ground shadow shrinks and the wall shadow grows. At 45° they're equal: both are 1.
The two shadows are always reciprocals: tan × cot = 1.

## 2 · All six on the turning arrow

Now the arrow of length one, on its circle. Its horizontal projection is cos, and its vertical projection is sin.
Extend the arrow's line until it meets the upright line **x = 1**: the height where it meets is **tan**, and the
distance along the line is **sec**. Extend it to the level line **y = 1**: the place where it meets is **cot**, and
the distance is **cosec**.

For angles where the arrow points away from one of these lines, its line still meets it behind the centre. That's where
the negative values come from.

In [ ]:
def six(angle_deg=35.0):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    fig, ax = plt.subplots(figsize=(7.2, 7.2))
    circle_axes(ax, lim=2.6)
    ax.axvline(1, color=DIM, lw=1.2, ls="--"); ax.axhline(1, color=DIM, lw=1.2, ls="--")
    ax.text(1.04, -2.45, "x = 1", color=DIM); ax.text(-2.5, 1.05, "y = 1", color=DIM)
    vals = []
    if abs(c) > 1e-3:
        t = s / c
        ax.plot([0, 1], [0, t], color=TEAL, lw=1.2, ls=":")
        ax.plot([1, 1], [0, t], color=TEAL, lw=6, solid_capstyle="butt")
        vals += [f"tan = {t:+.3f}", f"sec = {1 / c:+.3f}"]
    else:
        vals += ["tan: undefined (line parallel to x = 1)", "sec: undefined"]
    if abs(s) > 1e-3:
        k = c / s
        ax.plot([0, k], [0, 1], color=LILAC, lw=1.2, ls=":")
        ax.plot([0, k], [1, 1], color=LILAC, lw=6, solid_capstyle="butt")
        vals += [f"cot = {k:+.3f}", f"cosec = {1 / s:+.3f}"]
    else:
        vals += ["cot: undefined (line parallel to y = 1)", "cosec: undefined"]
    ax.plot([0, c], [0, 0], color=CHALK, lw=4, alpha=0.6); ax.plot([0, 0], [0, s], color=SAGE, lw=4, alpha=0.8)
    arrow(ax, a)
    ax.set_title(f"{angle_deg:.0f}°")
    ax.text(-2.5, -1.9, f"sin = {s:+.3f}   cos = {c:+.3f}\n" + "   ".join(vals[:2]) + "\n" + "   ".join(vals[2:]),
            fontsize=11.5, color=CHALK, va="top")
    plt.show()

interact(six, angle_deg=FloatSlider(value=35, min=-180, max=360, step=5, description="angle (°)"));

**Notice.** Drag the angle towards 90°. The line gets steeper, and the point where it meets x = 1 climbs out of
sight. **tan breaks at 90°**: a line parallel to x = 1 never meets it. Then drag to 135°: the line meets
x = 1 behind the centre, below the horizontal axis, so tan 135° = −1.

## 3 · Signs, domains and ranges

Each of the four new functions is a ratio of the two projections:
tan = sin ÷ cos, cot = cos ÷ sin, sec = 1 ÷ cos, cosec = 1 ÷ sin.
So each one breaks where its bottom is zero, and takes its signs from sin and cos.

In [ ]:
rows = [("sin", "all real x", "[−1, 1]"), ("cos", "all real x", "[−1, 1]"),
        ("tan", "x ≠ odd multiples of π/2", "all real numbers"), ("cot", "x ≠ multiples of π", "all real numbers"),
        ("sec", "x ≠ odd multiples of π/2", "(−∞, −1] ∪ [1, ∞)"), ("cosec", "x ≠ multiples of π", "(−∞, −1] ∪ [1, ∞)")]
print(f"{'':7}{'domain':30}{'range':22}   signs in quarters I  II  III  IV")
for name, dom, rng_ in rows:
    f = {"sin": np.sin, "cos": np.cos, "tan": np.tan, "cot": lambda v: 1 / np.tan(v),
         "sec": lambda v: 1 / np.cos(v), "cosec": lambda v: 1 / np.sin(v)}[name]
    signs = "   ".join("+" if f(math.radians(d)) > 0 else "−" for d in (45, 135, 225, 315))
    print(f"{name:7}{dom:30}{rng_:22}                    {signs}")

## 4 · Where tan breaks

Get close to 90° and watch tan grow without end. Just past 90°, it jumps to huge negative values: the line now meets
x = 1 far below.

In [ ]:
def near_90(gap_deg=10.0):
    x = np.linspace(-math.pi / 2 + 0.02, 3 * math.pi / 2 - 0.02, 1200)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(x, gaps(np.tan(x), 12), color=TEAL)
    for k in (-0.5, 0.5, 1.5): ax.axvline(k * math.pi, color=DIM, ls="--", lw=1)
    for d in (90 - gap_deg, 90 + gap_deg):
        v = math.tan(math.radians(d))
        if abs(v) < 12: ax.plot([math.radians(d)], [v], "o", color=CORAL, ms=9)
    ax.set_ylim(-12, 12); pi_ticks(ax, -math.pi / 2, 3 * math.pi / 2)
    ax.set_title(f"tan {90 - gap_deg:.1f}° = {math.tan(math.radians(90 - gap_deg)):+.2f},   "
                 f"tan {90 + gap_deg:.1f}° = {math.tan(math.radians(90 + gap_deg)):+.2f}")
    plt.show()

interact(near_90, gap_deg=FloatSlider(value=10, min=0.5, max=40, step=0.5, description="from 90° by"));

## 5 · Pythagoras, twice more

The arrow and its two projections made a right triangle with long side 1. That gave sin² + cos² = 1.

The line x = 1 makes another right triangle: one side 1 across, the other side tan, and the long side sec.
So **1 + tan² x = sec² x**. The ceiling y = 1 makes a third: sides 1 and cot, long side cosec. So
**1 + cot² x = cosec² x**.

In [ ]:
def pythagoras_twice(angle_deg=35.0):
    a = math.radians(angle_deg)
    c, s = math.cos(a), math.sin(a)
    t, k = s / c, c / s
    fig, ax = plt.subplots(figsize=(6.6, 6.6))
    circle_axes(ax, lim=2.2)
    ax.fill([0, 1, 1], [0, 0, t], color=TEAL, alpha=0.22)
    ax.fill([0, 0, k], [0, 1, 1], color=LILAC, alpha=0.22)
    ax.plot([0, 1, 1, 0], [0, 0, t, 0], color=TEAL, lw=2); ax.plot([0, 0, k, 0], [0, 1, 1, 0], color=LILAC, lw=2)
    arrow(ax, a)
    ax.set_title(f"1 + tan² = {1 + t * t:.4f} = sec² = {1 / c ** 2:.4f}\n"
                 f"1 + cot² = {1 + k * k:.4f} = cosec² = {1 / s ** 2:.4f}", fontsize=12)
    plt.show()

interact(pythagoras_twice, angle_deg=FloatSlider(value=35, min=10, max=80, step=1, description="angle (°)"));

## 6 · The identity gallery

**tan x = sin x ÷ cos x** and **cot x = cos x ÷ sin x.** The line x = 1 makes a triangle the same shape as the arrow's
own, scaled up by 1 ÷ cos x.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 2000)
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.6))
both_sides(axs[0], x, np.tan(x), np.sin(x) / np.cos(x), "tan x  and  sin x ÷ cos x", ylim=(-6, 6))
both_sides(axs[1], x, 1 / np.tan(x), np.cos(x) / np.sin(x), "cot x  and  cos x ÷ sin x", ylim=(-6, 6))
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(np.tan, lambda x: np.sin(x) / np.cos(x), ok=away_from_cos0)
check_identity(lambda x: 1 / np.tan(x), lambda x: np.cos(x) / np.sin(x), ok=away_from_sin0);

**sec x = 1 ÷ cos x** and **cosec x = 1 ÷ sin x.** The cutting lines' lengths, measured along the arrow's line.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.6))
sec_len = lambda v: np.hypot(1, np.tan(v)) * np.sign(np.cos(v))      # the distance to x = 1, with its sign
cosec_len = lambda v: np.hypot(1, 1 / np.tan(v)) * np.sign(np.sin(v))
both_sides(axs[0], x, sec_len(x), 1 / np.cos(x), "sec x (a length)  and  1 ÷ cos x", ylim=(-6, 6))
both_sides(axs[1], x, cosec_len(x), 1 / np.sin(x), "cosec x (a length)  and  1 ÷ sin x", ylim=(-6, 6))
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(sec_len, lambda x: 1 / np.cos(x), ok=away_from_cos0)
check_identity(cosec_len, lambda x: 1 / np.sin(x), ok=away_from_sin0);

**1 + tan² x = sec² x** and **1 + cot² x = cosec² x.** Pythagoras in the two new triangles.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.6))
both_sides(axs[0], x, 1 + np.tan(x) ** 2, 1 / np.cos(x) ** 2, "1 + tan² x  and  sec² x", ylim=(0, 8))
both_sides(axs[1], x, 1 + 1 / np.tan(x) ** 2, 1 / np.sin(x) ** 2, "1 + cot² x  and  cosec² x", ylim=(0, 8))
axs[0].legend(loc="upper right"); plt.tight_layout(); plt.show()
check_identity(lambda x: 1 + np.tan(x) ** 2, lambda x: 1 / np.cos(x) ** 2, ok=away_from_cos0)
check_identity(lambda x: 1 + 1 / np.tan(x) ** 2, lambda x: 1 / np.sin(x) ** 2, ok=away_from_sin0);

**tan(−x) = −tan x** and **tan(x + π) = tan x.** Turning the other way flips the meeting point. Half a turn points
the arrow backwards along the same line, so it meets x = 1 at the same place: tan repeats every π.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.6))
both_sides(axs[0], x, np.tan(-x), -np.tan(x), "tan(−x)  and  −tan x", ylim=(-6, 6))
both_sides(axs[1], x, np.tan(x + np.pi), np.tan(x), "tan(x + π)  and  tan x", ylim=(-6, 6))
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(lambda x: np.tan(-x), lambda x: -np.tan(x), ok=away_from_cos0)
check_identity(lambda x: np.tan(x + np.pi), np.tan, ok=away_from_cos0);

## 7 · On paper: the CBSE way

**Example 1.** If cos x = −3/5 and x is in the third quarter, find the other five.

1. sin² x = 1 − 9/25 = 16/25. In the third quarter the vertical projection points down: **sin x = −4/5**.
2. tan x = sin ÷ cos = (−4/5) ÷ (−3/5) = **4/3**, and **cot x = 3/4**.
3. **sec x = −5/3** and **cosec x = −5/4**.

In [ ]:
x1 = math.pi + math.acos(3 / 5)          # third quarter, cos = −3/5
print(f"sin {math.sin(x1):+.4f}  tan {math.tan(x1):+.4f}  cot {1 / math.tan(x1):+.4f}  "
      f"sec {1 / math.cos(x1):+.4f}  cosec {1 / math.sin(x1):+.4f}")

**Example 2.** Find tan(19π/3).

1. tan repeats every π, so take away whole halves: 19π/3 = 6π + π/3.
2. **tan(19π/3) = tan(π/3) = √3.**

In [ ]:
print(f"tan(19π/3) = {math.tan(19 * math.pi / 3):.6f},  √3 = {math.sqrt(3):.6f}")

**Try it.**
1. cot x = 3/4 and x is in the third quarter. Find sin x and cos x.
2. sec x = 13/5 and x is in the fourth quarter. Find sin x and tan x.

<details><summary>Answers</summary>

1. The ratio cos ÷ sin is 3/4, and both are negative: sin x = −4/5, cos x = −3/5.
2. cos x = 5/13; sin² x = 1 − 25/169 = 144/169, negative in the fourth quarter: sin x = −12/13, tan x = −12/5.

</details>

## Check yourself
1. Where is tan x undefined?
2. Can sec x equal 0.5?
3. A stick 2 m tall stands in the sun at 45°. How long is its shadow?

<details><summary>Answers</summary>

1. At odd multiples of π/2 (90°, 270° …), where the arrow's line is parallel to the line x = 1.
2. No. sec is the distance to the line x = 1, measured along the arrow's line, never shorter than 1: sec x ≤ −1 or sec x ≥ 1.
3. 2 m: at 45°, cot = 1.

</details>

## Sources
MacTutor History of Mathematics (St Andrews): "Thomas Fincke"; "Al-Biruni"; Miller, "Earliest uses of some words of
mathematics" (t). Museo Galileo, "Shadow square". NCERT class 11, chapter 3. CBSE Mathematics syllabus 2026–27. All
retrieved 2026-10-21.